# PatchCore WRN50 120k Local Results Review

This notebook loads the local artifact bundle for the labeled `120k / 10k / 20k` PatchCore WideResNet50-2 run and summarizes the final evaluation.

It is meant to be a lightweight review notebook for:
- the selected variant and threshold
- the comparison sweep across WRN50 PatchCore settings
- held-out test metrics on the `20k` evaluation split
- the main false positive and false negative tables

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

cwd = Path.cwd().resolve()
EXPERIMENT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "helpers" / "patchcore_wrn50_local.py").exists() and (candidate / "README.md").exists():
        EXPERIMENT_ROOT = candidate
        break

if EXPERIMENT_ROOT is None:
    raise RuntimeError("Could not locate the local WRN PatchCore experiment root.")

ARTIFACT_DIR = EXPERIMENT_ROOT / "artifacts" / "patchcore_wrn50_multilayer_120k_5pct"

if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(
        f"No local artifact bundle was found at {ARTIFACT_DIR}. Run notebook.ipynb first to generate it."
    )

In [ ]:
summary = json.loads((ARTIFACT_DIR / "bundle_summary.json").read_text(encoding="utf-8"))
sweep_df = pd.read_csv(ARTIFACT_DIR / "patchcore_sweep_results.csv").sort_values("f1", ascending=False).reset_index(drop=True)
selected_variant = str(summary["selected_variant"])
selected_row = sweep_df.loc[sweep_df["name"] == selected_variant].iloc[0]
predictions_path = ARTIFACT_DIR / f"{selected_variant}_test_predictions.csv"
pred_df = pd.read_csv(predictions_path)

run_info_df = pd.DataFrame(
    [
        {"item": "selected_variant", "value": selected_variant},
        {"item": "validation_threshold", "value": f"{selected_row['threshold']:.6f}"},
        {"item": "experiment_root", "value": str(EXPERIMENT_ROOT)},
        {"item": "artifact_dir", "value": str(ARTIFACT_DIR)},
        {"item": "predictions_path", "value": str(predictions_path)},
    ]
)
split_config_df = pd.DataFrame(summary["split_config"].items(), columns=["split_setting", "value"])
sweep_view_df = sweep_df[
    [
        "name",
        "reduction",
        "topk_ratio",
        "threshold",
        "precision",
        "recall",
        "f1",
        "auroc",
        "auprc",
        "best_sweep_threshold",
        "best_sweep_f1",
        "predicted_anomalies",
    ]
].copy()

display(run_info_df)
display(split_config_df)
display(sweep_view_df)

In [ ]:
y_true = pred_df["is_anomaly"].astype(int)
y_pred = pred_df["predicted_is_anomaly"].astype(int)
y_score = pred_df["score"].astype(float)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

metrics_df = pd.DataFrame(
    [
        {"metric": "threshold", "value": float(selected_row["threshold"])},
        {"metric": "precision", "value": precision_score(y_true, y_pred, zero_division=0)},
        {"metric": "recall", "value": recall_score(y_true, y_pred, zero_division=0)},
        {"metric": "f1", "value": f1_score(y_true, y_pred, zero_division=0)},
        {"metric": "accuracy", "value": accuracy_score(y_true, y_pred)},
        {"metric": "balanced_accuracy", "value": balanced_accuracy_score(y_true, y_pred)},
        {"metric": "auroc", "value": roc_auc_score(y_true, y_score)},
        {"metric": "auprc", "value": average_precision_score(y_true, y_score)},
        {"metric": "true_negatives", "value": int(tn)},
        {"metric": "false_positives", "value": int(fp)},
        {"metric": "false_negatives", "value": int(fn)},
        {"metric": "true_positives", "value": int(tp)},
        {"metric": "predicted_anomalies", "value": int(y_pred.sum())},
    ]
)

error_summary_df = pred_df["error_type"].value_counts().rename_axis("error_type").reset_index(name="count")
false_negative_by_type_df = (
    pred_df[(pred_df["is_anomaly"] == 1) & (pred_df["predicted_is_anomaly"] == 0)]["defect_type"]
    .value_counts()
    .rename_axis("defect_type")
    .reset_index(name="count")
)

display(metrics_df)
display(error_summary_df)
display(false_negative_by_type_df.head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

variant_colors = ["#0a9396" if name == selected_variant else "#94d2bd" for name in sweep_df["name"]]
axes[0].bar(sweep_df["name"], sweep_df["f1"], color=variant_colors)
axes[0].set_title("Held-out Test F1 by Variant")
axes[0].set_ylabel("F1")
axes[0].tick_params(axis="x", rotation=20)

cm = [[tn, fp], [fn, tp]]
image = axes[1].imshow(cm, cmap="Blues")
axes[1].set_title("Confusion Matrix")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Pred Normal", "Pred Anomaly"])
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["True Normal", "True Anomaly"])
for row_index in range(2):
    for col_index in range(2):
        axes[1].text(col_index, row_index, f"{cm[row_index][col_index]:,}", ha="center", va="center", color="black")
plt.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].hist(y_score[y_true == 0], bins=40, alpha=0.75, label="normal", color="#94d2bd")
axes[2].hist(y_score[y_true == 1], bins=40, alpha=0.75, label="anomaly", color="#ee9b00")
axes[2].axvline(float(selected_row["threshold"]), color="red", linestyle="--", label=f"threshold={selected_row['threshold']:.4f}")
axes[2].set_title(f"Score Distribution: {selected_variant}")
axes[2].set_xlabel("PatchCore score")
axes[2].set_ylabel("Count")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
false_positive_df = pred_df[(pred_df["is_anomaly"] == 0) & (pred_df["predicted_is_anomaly"] == 1)].sort_values("score", ascending=False)
false_negative_df = pred_df[(pred_df["is_anomaly"] == 1) & (pred_df["predicted_is_anomaly"] == 0)].sort_values("score", ascending=True)

print("Top false positives")
display(false_positive_df[["array_path", "source_split", "score"]].head(10))

print("Top false negatives")
display(false_negative_df[["array_path", "defect_type", "source_split", "score"]].head(10))